# 01 — Official MobileGeo Release Audit

**Objective:** Clone and pin the official MobileGeo repository, inventory released code, checkpoints, features, preprocessing, evaluator and licensing.

**Project:** GPS-Denied UAV Visual Positioning System

**Provenance rule:** Never describe local reimplementation results as official MobileGeo results.


In [ ]:
from pathlib import Path
import json
import hashlib
import random
import numpy as np

PROJECT_ROOT = Path("/content/drive/MyDrive/mobilegeo_project")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("Project root:", PROJECT_ROOT)
print("Seed:", SEED)


In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

AUDIT_ROOT = (
    PROJECT_ROOT / "results" / "mobilegeo_audit"
)

REPO_ROOT = (
    PROJECT_ROOT / "repositories"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPO_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Audit output:", AUDIT_ROOT)
print("Repository folder:", REPO_ROOT)
print("✅ Audit setup ready")

Mounted at /content/drive
Project root: /content/drive/MyDrive/mobilegeo_project
Audit output: /content/drive/MyDrive/mobilegeo_project/results/mobilegeo_audit
Repository folder: /content/drive/MyDrive/mobilegeo_project/repositories
✅ Audit setup ready


In [2]:
# ============================================================
# AUDIT CELL 2
# Clone the official MobileGeo repository and pin commit hash
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import shutil
import json

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mobilegeo_audit"
)

REPOSITORIES_ROOT = (
    PROJECT_ROOT
    / "repositories"
)

MOBILEGEO_REPO = (
    REPOSITORIES_ROOT
    / "MobileGeo"
)

REPOSITORY_URL = (
    "https://github.com/SkyEyeLoc/MobileGeo.git"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

REPOSITORIES_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def run_command(
    command,
    cwd=None,
):
    result = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        print("STDOUT:")
        print(result.stdout)

        print("STDERR:")
        print(result.stderr)

        raise RuntimeError(
            "Command failed:\n"
            + " ".join(command)
        )

    return result.stdout.strip()


# Clone only when repository does not exist.
if not (
    MOBILEGEO_REPO
    / ".git"
).is_dir():

    if MOBILEGEO_REPO.exists():
        shutil.rmtree(
            MOBILEGEO_REPO
        )

    print("Cloning official MobileGeo repository...")

    run_command(
        [
            "git",
            "clone",
            REPOSITORY_URL,
            str(MOBILEGEO_REPO),
        ]
    )

else:
    print(
        "Repository already exists. "
        "Using the existing clone."
    )

# Record repository state.
commit_hash = run_command(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=MOBILEGEO_REPO,
)

short_commit = run_command(
    [
        "git",
        "rev-parse",
        "--short",
        "HEAD",
    ],
    cwd=MOBILEGEO_REPO,
)

branch_name = run_command(
    [
        "git",
        "branch",
        "--show-current",
    ],
    cwd=MOBILEGEO_REPO,
)

remote_url = run_command(
    [
        "git",
        "remote",
        "get-url",
        "origin",
    ],
    cwd=MOBILEGEO_REPO,
)

commit_date = run_command(
    [
        "git",
        "show",
        "-s",
        "--format=%cI",
        "HEAD",
    ],
    cwd=MOBILEGEO_REPO,
)

commit_subject = run_command(
    [
        "git",
        "show",
        "-s",
        "--format=%s",
        "HEAD",
    ],
    cwd=MOBILEGEO_REPO,
)

repository_record = {
    "audit_created_at": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "repository_name": "MobileGeo",
    "repository_url": remote_url,
    "local_path": str(
        MOBILEGEO_REPO
    ),
    "branch": branch_name,
    "commit_hash": commit_hash,
    "short_commit": short_commit,
    "commit_date": commit_date,
    "commit_subject": commit_subject,
    "audit_status": (
        "REPOSITORY_CLONED_NOT_YET_CLASSIFIED"
    ),
}

repository_record_path = (
    AUDIT_ROOT
    / "repository_record.json"
)

repository_record_path.write_text(
    json.dumps(
        repository_record,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n✅ Official repository ready")

print("\nRepository:")
print(MOBILEGEO_REPO)

print("\nBranch:")
print(branch_name)

print("\nPinned commit:")
print(commit_hash)

print("\nCommit date:")
print(commit_date)

print("\nCommit subject:")
print(commit_subject)

print("\nSaved repository record:")
print(repository_record_path)

Cloning official MobileGeo repository...

✅ Official repository ready

Repository:
/content/drive/MyDrive/mobilegeo_project/repositories/MobileGeo

Branch:
main

Pinned commit:
e17eafc070b44116e91880c45e45d1c21fc36287

Commit date:
2025-11-10T09:59:08+08:00

Commit subject:
final

Saved repository record:
/content/drive/MyDrive/mobilegeo_project/results/mobilegeo_audit/repository_record.json


In [3]:
# ============================================================
# AUDIT CELL 3
# Inventory every released repository file
# ============================================================

from pathlib import Path
import pandas as pd
import hashlib
import os

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mobilegeo_audit"
)

MOBILEGEO_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "MobileGeo"
)

if not MOBILEGEO_REPO.is_dir():
    raise FileNotFoundError(
        f"Repository not found:\n{MOBILEGEO_REPO}"
    )


def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


records = []

for file_path in sorted(
    MOBILEGEO_REPO.rglob("*")
):
    if not file_path.is_file():
        continue

    # Do not inventory internal Git objects.
    if ".git" in file_path.parts:
        continue

    relative_path = file_path.relative_to(
        MOBILEGEO_REPO
    )

    suffix = (
        file_path.suffix.lower()
        if file_path.suffix
        else "[no_extension]"
    )

    records.append({
        "relative_path": str(
            relative_path
        ),
        "filename": file_path.name,
        "extension": suffix,
        "size_bytes": file_path.stat().st_size,
        "sha256": sha256_file(
            file_path
        ),
    })

inventory_df = pd.DataFrame(
    records
)

if inventory_df.empty:
    raise RuntimeError(
        "Repository inventory is empty."
    )

inventory_csv = (
    AUDIT_ROOT
    / "repository_file_inventory.csv"
)

inventory_df.to_csv(
    inventory_csv,
    index=False,
)

extension_summary = (
    inventory_df
    .groupby(
        "extension",
        dropna=False,
    )
    .agg(
        file_count=(
            "relative_path",
            "count",
        ),
        total_size_bytes=(
            "size_bytes",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "file_count",
            "extension",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

extension_summary_csv = (
    AUDIT_ROOT
    / "repository_extension_summary.csv"
)

extension_summary.to_csv(
    extension_summary_csv,
    index=False,
)

print("✅ Repository inventory created")

print("\nTotal released files:")
print(len(inventory_df))

print("\nTotal released size:")
print(
    f"{inventory_df['size_bytes'].sum() / (1024 * 1024):.2f} MB"
)

print("\nTop-level repository items:")

for path in sorted(
    MOBILEGEO_REPO.iterdir()
):
    if path.name == ".git":
        continue

    item_type = (
        "DIR"
        if path.is_dir()
        else "FILE"
    )

    print(
        f"- [{item_type}] {path.name}"
    )

print("\nExtension summary:")
print(
    extension_summary.to_string(
        index=False
    )
)

print("\nInventory CSV:")
print(inventory_csv)

print("\nExtension summary CSV:")
print(extension_summary_csv)

✅ Repository inventory created

Total released files:
33

Total released size:
2.68 MB

Top-level repository items:
- [DIR] FoundationModel
- [FILE] README.md
- [FILE] README.md~
- [DIR] Tools
- [DIR] assets
- [FILE] requirements.txt

Extension summary:
extension  file_count  total_size_bytes
     .pyc          18            158469
      .py           9             64301
     .png           2           1856890
    .jpeg           1            721858
      .md           1              5463
     .md~           1              4721
     .txt           1              1156

Inventory CSV:
/content/drive/MyDrive/mobilegeo_project/results/mobilegeo_audit/repository_file_inventory.csv

Extension summary CSV:
/content/drive/MyDrive/mobilegeo_project/results/mobilegeo_audit/repository_extension_summary.csv


In [4]:
# ============================================================
# AUDIT CELL 4
# Classify released model, checkpoint, feature and evaluator files
# ============================================================

from pathlib import Path
import pandas as pd
import re
import json

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mobilegeo_audit"
)

MOBILEGEO_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "MobileGeo"
)

inventory_csv = (
    AUDIT_ROOT
    / "repository_file_inventory.csv"
)

if not inventory_csv.is_file():
    raise FileNotFoundError(
        "Run AUDIT CELL 3 first."
    )

inventory_df = pd.read_csv(
    inventory_csv
)

categories = {
    "python_code": [
        r"\.py$",
    ],

    "checkpoint_or_weight": [
        r"\.pth$",
        r"\.pt$",
        r"\.ckpt$",
        r"\.bin$",
        r"\.safetensors$",
        r"weight",
        r"checkpoint",
        r"model_",
    ],

    "precomputed_feature": [
        r"\.mat$",
        r"\.npy$",
        r"\.npz$",
        r"feature",
        r"descriptor",
        r"embedding",
    ],

    "evaluation_code": [
        r"evaluat",
        r"metric",
        r"recall",
        r"rank",
        r"test",
    ],

    "training_code": [
        r"train",
        r"loss",
        r"optim",
        r"distill",
    ],

    "model_definition": [
        r"model",
        r"network",
        r"backbone",
        r"mobile",
        r"foundation",
    ],

    "preprocessing": [
        r"transform",
        r"normalize",
        r"resize",
        r"dataset",
        r"dataloader",
        r"preprocess",
    ],

    "configuration": [
        r"\.yaml$",
        r"\.yml$",
        r"\.toml$",
        r"\.json$",
        r"config",
        r"requirement",
    ],

    "license": [
        r"license",
        r"copying",
    ],
}

classification_records = []

for _, row in inventory_df.iterrows():

    relative_path = str(
        row["relative_path"]
    )

    search_text = (
        relative_path.lower()
    )

    matched_categories = []

    for category, patterns in categories.items():

        if any(
            re.search(
                pattern,
                search_text,
                flags=re.IGNORECASE,
            )
            for pattern in patterns
        ):
            matched_categories.append(
                category
            )

    if not matched_categories:
        matched_categories = [
            "other"
        ]

    for category in matched_categories:
        classification_records.append({
            "category": category,
            "relative_path": relative_path,
            "size_bytes": int(
                row["size_bytes"]
            ),
        })

classification_df = pd.DataFrame(
    classification_records
)

classification_csv = (
    AUDIT_ROOT
    / "repository_asset_classification.csv"
)

classification_df.to_csv(
    classification_csv,
    index=False,
)

category_summary = (
    classification_df
    .groupby("category")
    .agg(
        file_count=(
            "relative_path",
            "nunique",
        )
    )
    .reset_index()
    .sort_values(
        "category"
    )
)

category_summary_csv = (
    AUDIT_ROOT
    / "repository_asset_summary.csv"
)

category_summary.to_csv(
    category_summary_csv,
    index=False,
)

important_categories = [
    "checkpoint_or_weight",
    "precomputed_feature",
    "model_definition",
    "evaluation_code",
    "training_code",
    "preprocessing",
    "license",
]

print("✅ Repository assets classified")

print("\nCategory summary:")
print(
    category_summary.to_string(
        index=False
    )
)

for category in important_categories:

    matching_paths = (
        classification_df[
            classification_df[
                "category"
            ] == category
        ]["relative_path"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    print(
        "\n"
        + "=" * 60
    )

    print(
        category.upper()
    )

    print(
        "=" * 60
    )

    if not matching_paths:
        print(
            "No matching repository files found."
        )

    else:
        for path in matching_paths:
            print(
                "-",
                path,
            )

# Initial evidence record.
initial_evidence = {
    "repository_audited": True,
    "raw_image_inference_tested": False,
    "official_checkpoint_verified": False,
    "precomputed_features_verified": False,
    "licence_verified": False,
    "final_provenance_decision": (
        "PENDING_FURTHER_AUDIT"
    ),
}

evidence_path = (
    AUDIT_ROOT
    / "initial_release_evidence.json"
)

evidence_path.write_text(
    json.dumps(
        initial_evidence,
        indent=2,
    ),
    encoding="utf-8",
)

print("\nClassification CSV:")
print(classification_csv)

print("\nInitial evidence record:")
print(evidence_path)

✅ Repository assets classified

Category summary:
            category  file_count
checkpoint_or_weight           2
       configuration           1
     evaluation_code           1
    model_definition          26
               other           5
         python_code           9

CHECKPOINT_OR_WEIGHT
- FoundationModel/backbones/__pycache__/model_convnext.cpython-312.pyc
- FoundationModel/backbones/model_convnext.py

PRECOMPUTED_FEATURE
No matching repository files found.

MODEL_DEFINITION
- FoundationModel/backbones/ConvNeXt.py
- FoundationModel/backbones/ConvNeXt_bn.py
- FoundationModel/backbones/__init__.py
- FoundationModel/backbones/__pycache__/ConvNeXt.cpython-311.pyc
- FoundationModel/backbones/__pycache__/ConvNeXt.cpython-312.pyc
- FoundationModel/backbones/__pycache__/ConvNeXt_bn.cpython-311.pyc
- FoundationModel/backbones/__pycache__/ConvNeXt_bn.cpython-312.pyc
- FoundationModel/backbones/__pycache__/__init__.cpython-310.pyc
- FoundationModel/backbones/__pycache__/__init__.cp

In [5]:
# ============================================================
# AUDIT CELL 5
# Remove cache files and identify real checkpoints/features
# ============================================================

from pathlib import Path
import pandas as pd
import hashlib

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

MOBILEGEO_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "MobileGeo"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mobilegeo_audit"
)

CHECKPOINT_EXTENSIONS = {
    ".pth",
    ".pt",
    ".ckpt",
    ".bin",
    ".safetensors",
    ".onnx",
    ".engine",
}

FEATURE_EXTENSIONS = {
    ".mat",
    ".npy",
    ".npz",
    ".h5",
    ".hdf5",
}

ARCHIVE_EXTENSIONS = {
    ".zip",
    ".tar",
    ".gz",
    ".7z",
    ".rar",
}


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(1024 * 1024)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


records = []

for path in sorted(MOBILEGEO_REPO.rglob("*")):

    if not path.is_file():
        continue

    relative = path.relative_to(MOBILEGEO_REPO)

    # Ignore Git internals and Python caches.
    if ".git" in relative.parts:
        continue

    if "__pycache__" in relative.parts:
        continue

    if path.suffix.lower() == ".pyc":
        continue

    suffix = path.suffix.lower()

    if suffix in CHECKPOINT_EXTENSIONS:
        asset_type = "checkpoint_or_export"

    elif suffix in FEATURE_EXTENSIONS:
        asset_type = "precomputed_feature"

    elif suffix in ARCHIVE_EXTENSIONS:
        asset_type = "archive"

    elif suffix == ".py":
        asset_type = "python_source"

    elif path.name.lower() in {
        "license",
        "license.txt",
        "license.md",
        "copying",
    }:
        asset_type = "license"

    else:
        asset_type = "other"

    records.append({
        "relative_path": str(relative),
        "filename": path.name,
        "extension": suffix or "[none]",
        "asset_type": asset_type,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

clean_df = pd.DataFrame(records)

clean_inventory_path = (
    AUDIT_ROOT
    / "repository_clean_inventory.csv"
)

clean_df.to_csv(
    clean_inventory_path,
    index=False,
)

summary = (
    clean_df
    .groupby("asset_type")
    .agg(
        file_count=("relative_path", "count"),
        total_size_bytes=("size_bytes", "sum"),
    )
    .reset_index()
)

print("✅ Clean repository inventory created")

print("\nAsset summary:")
print(summary.to_string(index=False))

for asset_type in [
    "checkpoint_or_export",
    "precomputed_feature",
    "archive",
    "license",
]:

    selected = clean_df[
        clean_df["asset_type"] == asset_type
    ]

    print("\n" + "=" * 60)
    print(asset_type.upper())
    print("=" * 60)

    if selected.empty:
        print("None found.")
    else:
        for path in selected["relative_path"]:
            print("-", path)

print("\nLargest released files:")

largest = (
    clean_df
    .sort_values("size_bytes", ascending=False)
    .head(20)
)

print(
    largest[
        [
            "relative_path",
            "asset_type",
            "size_bytes",
        ]
    ].to_string(index=False)
)

print("\nSaved:")
print(clean_inventory_path)

✅ Clean repository inventory created

Asset summary:
   asset_type  file_count  total_size_bytes
        other           6           2590088
python_source           9             64301

CHECKPOINT_OR_EXPORT
None found.

PRECOMPUTED_FEATURE
None found.

ARCHIVE
None found.

LICENSE
None found.

Largest released files:
                              relative_path    asset_type  size_bytes
                      assets/frame_work.png         other     1260680
                            assets/tx2.jpeg         other      721858
                      assets/bubble_all.png         other      596210
         FoundationModel/backbones/osnet.py python_source       17147
      FoundationModel/backbones/ConvNeXt.py python_source       11773
FoundationModel/backbones/model_convnext.py python_source        9834
   FoundationModel/backbones/ConvNeXt_bn.py python_source        9824
                                  README.md         other        5463
        FoundationModel/backbones/dinov2.py python_

In [6]:
# ============================================================
# AUDIT CELL 6
# Print important released text and source files
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

MOBILEGEO_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "MobileGeo"
)

important_names = {
    "readme",
    "requirements.txt",
    "environment.yml",
    "environment.yaml",
    "setup.py",
    "pyproject.toml",
    "license",
    "license.txt",
    "license.md",
}

important_paths = []

for path in MOBILEGEO_REPO.rglob("*"):

    if not path.is_file():
        continue

    if ".git" in path.parts:
        continue

    if "__pycache__" in path.parts:
        continue

    name_lower = path.name.lower()

    if (
        name_lower in important_names
        or name_lower.startswith("readme")
        or path.name == "evaluate_norm.py"
    ):
        important_paths.append(path)

print("Important files found:", len(important_paths))

for path in sorted(important_paths):

    print("\n" + "=" * 80)
    print(path.relative_to(MOBILEGEO_REPO))
    print("=" * 80)

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="replace",
        )

        # Limit very long files.
        lines = text.splitlines()

        for line_number, line in enumerate(
            lines[:400],
            start=1,
        ):
            print(
                f"{line_number:04d}: {line}"
            )

        if len(lines) > 400:
            print(
                f"\n... truncated; total lines: {len(lines)}"
            )

    except Exception as error:
        print(
            "Could not read file:",
            error,
        )

Important files found: 4

README.md
0001: # 🌍 MobileGeo: Exploring Hierarchical Knowledge Distillation for Resource-Efficient Cross-view Drone Geo-Localization
0002: 
0003: 
0004: This is the official PyTorch implementation for our paper **"MobileGeo: Exploring Hierarchical Knowledge Distillation for Resource-Efficient Cross-view Drone Geo-Localization"**.
0005: 
0006: In this project, We propose **MobileGeo**, a resource-efficient framework combining hierarchical knowledge transfer and multi-view representation refinement. 🚁
0007: 
0008: <img src="assets/tx2.jpeg" width="90%" height="90%">
0009: 
0010: ---
0011: 
0012: ## 🌟 Efficiency analysis
0013: 
0014: **MobileGeo** achieves state-of-the-art performance in both accuracy and efficiency, reaching `97.15% Recall@1` on University-1652 while being over `5x` more efficient in `FLOPs` and `3x faster` than previous top methods. 
0015: 
0016: Furthermore, MobileGeo runs at `251.5 FPS` on the `AGX Orin` edge device, demonstrating its practi

In [7]:
# ============================================================
# AUDIT CELL 7
# Search source code for checkpoints, features, datasets,
# normalization, input size and download links
# ============================================================

from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

MOBILEGEO_REPO = (
    PROJECT_ROOT
    / "repositories"
    / "MobileGeo"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mobilegeo_audit"
)

search_patterns = {
    "checkpoint_reference": (
        r"\.(pth|pt|ckpt|bin|safetensors)\b"
        r"|checkpoint"
        r"|load_state_dict"
        r"|torch\.load"
    ),

    "feature_reference": (
        r"\.(mat|npy|npz|h5|hdf5)\b"
        r"|loadmat"
        r"|savemat"
    ),

    "input_resolution": (
        r"resize"
        r"|input_size"
        r"|image_size"
        r"|height"
        r"|width"
    ),

    "normalization": (
        r"normalize"
        r"|mean\s*="
        r"|std\s*="
    ),

    "dataset_reference": (
        r"SUES"
        r"|University"
        r"|drone"
        r"|satellite"
        r"|gallery"
        r"|query"
    ),

    "download_link": (
        r"https?://"
        r"|drive\.google"
        r"|huggingface"
        r"|baidu"
        r"|dropbox"
    ),

    "training_reference": (
        r"optimizer"
        r"|backward\("
        r"|loss"
        r"|train_loader"
        r"|epoch"
    ),

    "inference_reference": (
        r"model\.eval"
        r"|no_grad"
        r"|inference"
        r"|extract_feature"
        r"|forward"
    ),
}

records = []

for path in sorted(
    MOBILEGEO_REPO.rglob("*.py")
):

    if "__pycache__" in path.parts:
        continue

    relative_path = str(
        path.relative_to(MOBILEGEO_REPO)
    )

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    lines = text.splitlines()

    for category, pattern in search_patterns.items():

        regex = re.compile(
            pattern,
            flags=re.IGNORECASE,
        )

        for line_number, line in enumerate(
            lines,
            start=1,
        ):

            if regex.search(line):
                records.append({
                    "category": category,
                    "relative_path": relative_path,
                    "line_number": line_number,
                    "line_text": line.strip(),
                })

search_df = pd.DataFrame(records)

search_output = (
    AUDIT_ROOT
    / "source_code_evidence.csv"
)

search_df.to_csv(
    search_output,
    index=False,
)

print("✅ Source-code evidence search completed")

if search_df.empty:
    print("No matching evidence found.")

else:
    summary = (
        search_df
        .groupby("category")
        .size()
        .reset_index(name="match_count")
        .sort_values("category")
    )

    print("\nEvidence summary:")
    print(summary.to_string(index=False))

    for category in search_patterns:

        category_df = search_df[
            search_df["category"] == category
        ]

        print("\n" + "=" * 70)
        print(category.upper())
        print("=" * 70)

        if category_df.empty:
            print("No evidence found.")
            continue

        for _, row in category_df.head(30).iterrows():
            print(
                f"{row['relative_path']}:"
                f"{row['line_number']} — "
                f"{row['line_text']}"
            )

        if len(category_df) > 30:
            print(
                f"... {len(category_df) - 30} more matches "
                "saved in the CSV."
            )

print("\nSaved evidence CSV:")
print(search_output)

✅ Source-code evidence search completed

Evidence summary:
            category  match_count
checkpoint_reference           37
   dataset_reference           20
       download_link           35
   feature_reference            2
 inference_reference           31
    input_resolution           14
       normalization           22
  training_reference           15

CHECKPOINT_REFERENCE
FoundationModel/backbones/ConvNeXt.py:243 — # https://dl.fbaipublicfiles.com/convnext/convnext_tiny_1k_224_ema.pth
FoundationModel/backbones/ConvNeXt.py:251 — # https://dl.fbaipublicfiles.com/convnext/convnext_small_1k_224_ema.pth
FoundationModel/backbones/ConvNeXt.py:259 — # https://dl.fbaipublicfiles.com/convnext/convnext_base_1k_224_ema.pth
FoundationModel/backbones/ConvNeXt.py:260 — # https://dl.fbaipublicfiles.com/convnext/convnext_base_22k_224.pth
FoundationModel/backbones/ConvNeXt.py:268 — # https://dl.fbaipublicfiles.com/convnext/convnext_large_1k_224_ema.pth
FoundationModel/backbones/ConvNeXt.py:2

In [8]:
# ============================================================
# AUDIT CELL 8
# Save provisional official-release audit decision
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mobilegeo_audit"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

decision = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "repository": "SkyEyeLoc/MobileGeo",

    "repository_claims_official_implementation": True,

    "repository_checkpoint_found": False,

    "official_mobilegeo_weight_link_validated": False,

    "raw_image_inference_script_found": False,

    "complete_query_gallery_pipeline_found": False,

    "project_training_pipeline_found": False,

    "released_mat_evaluator_found": True,

    "released_mat_download_link_found": True,

    "released_mat_asset_verified": False,

    "license_file_found": False,

    "important_findings": [
        (
            "Tools/evaluate_norm.py evaluates precomputed "
            ".mat features."
        ),
        (
            "The evaluator expects query_f, query_label, "
            "gallery_f and gallery_label."
        ),
        (
            "No trained MobileGeo checkpoint was found "
            "inside the cloned repository."
        ),
        (
            "Backbone checkpoint URLs are generic ConvNeXt "
            "pretraining weights, not MobileGeo weights."
        ),
        (
            "The README MobileGeo weight link is empty."
        ),
        (
            "No complete raw-image preprocessing and "
            "query/gallery inference workflow was found."
        ),
        (
            "No repository license file was found."
        ),
        (
            "README.md~ contains older PFED text and must "
            "not be treated as the current MobileGeo release."
        ),
    ],

    "provisional_provenance_status": (
        "PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION_"
        "PENDING_ASSET_VERIFICATION"
    ),

    "official_checkpoint_reproduction_supported": False,

    "next_action": (
        "Download and validate the released .mat feature files."
    ),
}

decision_json = (
    AUDIT_ROOT
    / "provisional_provenance_decision.json"
)

decision_json.write_text(
    json.dumps(
        decision,
        indent=2,
    ),
    encoding="utf-8",
)

decision_markdown = (
    AUDIT_ROOT
    / "provisional_provenance_decision.md"
)

decision_markdown.write_text(
    """# MobileGeo Official Release Audit

## Provisional status

`PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION_PENDING_ASSET_VERIFICATION`

## Supported evidence

- The repository contains a `.mat` feature evaluator.
- The README publishes a Google Drive link for `.mat` files.
- No trained MobileGeo checkpoint was found in the repository.
- No complete raw-image inference pipeline was identified.
- The MobileGeo weights link in the README is empty.
- No repository licence file was identified.

## Current restriction

The project must not report an official MobileGeo raw-image
checkpoint reproduction.

## Next action

Download the released `.mat` assets and verify their schema.
""",
    encoding="utf-8",
)

print("✅ Provisional audit decision saved")

print("\nStatus:")
print(
    decision[
        "provisional_provenance_status"
    ]
)

print("\nJSON:")
print(decision_json)

print("\nMarkdown report:")
print(decision_markdown)

✅ Provisional audit decision saved

Status:
PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION_PENDING_ASSET_VERIFICATION

JSON:
/content/drive/MyDrive/mobilegeo_project/results/mobilegeo_audit/provisional_provenance_decision.json

Markdown report:
/content/drive/MyDrive/mobilegeo_project/results/mobilegeo_audit/provisional_provenance_decision.md


In [9]:
# ============================================================
# AUDIT CELL 9
# Download MobileGeo released precomputed .mat features
# ============================================================

from pathlib import Path
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mobilegeo_audit"
)

MAT_ASSET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "official_mobilegeo_precomputed_features"
)

MAT_ASSET_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

GOOGLE_DRIVE_FOLDER_URL = (
    "https://drive.google.com/drive/folders/"
    "1mmIp8HotaW0hBTC3zTxYKm1o_1ET8Ity"
    "?usp=drive_link"
)

# Install only gdown, not the repository requirements.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "gdown",
    ],
    check=True,
)

print("Downloading released MobileGeo feature assets...")

download_result = subprocess.run(
    [
        "gdown",
        "--folder",
        GOOGLE_DRIVE_FOLDER_URL,
        "-O",
        str(MAT_ASSET_ROOT),
    ],
    capture_output=True,
    text=True,
    check=False,
)

print("\nDownload stdout:")
print(download_result.stdout)

if download_result.stderr.strip():
    print("\nDownload stderr:")
    print(download_result.stderr)

mat_files = sorted(
    MAT_ASSET_ROOT.rglob("*.mat")
)

records = []

for mat_file in mat_files:
    records.append({
        "filename": mat_file.name,
        "relative_path": str(
            mat_file.relative_to(
                MAT_ASSET_ROOT
            )
        ),
        "absolute_path": str(mat_file),
        "size_bytes": mat_file.stat().st_size,
        "size_mb": (
            mat_file.stat().st_size
            / (1024 * 1024)
        ),
    })

mat_inventory_df = pd.DataFrame(
    records
)

mat_inventory_csv = (
    AUDIT_ROOT
    / "released_mat_asset_inventory.csv"
)

mat_inventory_df.to_csv(
    mat_inventory_csv,
    index=False,
)

print("\n" + "=" * 65)
print("RELEASED MAT ASSET RESULTS")
print("=" * 65)

print("\nDownload return code:")
print(download_result.returncode)

print("\nMAT files found:")
print(len(mat_files))

if mat_files:
    print(
        "\n",
        mat_inventory_df[
            [
                "relative_path",
                "size_mb",
            ]
        ].to_string(
            index=False
        ),
    )

else:
    print(
        "\nNo .mat files were downloaded."
    )

print("\nAsset folder:")
print(MAT_ASSET_ROOT)

print("\nInventory CSV:")
print(mat_inventory_csv)


Download stdout:
Processing file 1BD4SAHShv4WjMGOO6eV6temE_TAWyuB2 second_D2S.mat
Processing file 1Y9Mn5-v9OkyHVI8fioT2BMxqGhlj1EZk second_S2D.mat


Download stderr:
Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1BD4SAHShv4WjMGOO6eV6temE_TAWyuB2
From (redirected): https://drive.google.com/uc?id=1BD4SAHShv4WjMGOO6eV6temE_TAWyuB2&confirm=t&uuid=310f165c-45a5-459e-81ef-379b9869f314
To: /content/drive/MyDrive/mobilegeo_project/data/official_mobilegeo_precomputed_features/second_D2S.mat

  0%|          | 0.00/123M [00:00<?, ?B/s]
  4%|▍         | 4.72M/123M [00:00<00:07, 14.8MB/s]
  6%|▌         | 6.82M/123M [00:00<00:07, 15.6MB/s]
  9%|▉         | 11.0M/123M [00:00<00:05, 20.9MB/s]
 19%|█▉        | 23.6M/123M [00:00<00:03, 31.3MB/s]
 32%|███▏      | 38.8M/123M [00:01<00:01, 49.0MB/s]
 36%|███▌      | 44.6M/123M [00:01<00:01, 41.9MB/s]
 40%|██

In [10]:
# ============================================================
# AUDIT CELL 10
# Validate released .mat feature schema and assign provenance
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import scipy.io
import pandas as pd
import json

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mobilegeo_audit"
)

MAT_ASSET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "official_mobilegeo_precomputed_features"
)

required_variables = {
    "query_f",
    "query_label",
    "gallery_f",
    "gallery_label",
}

mat_files = sorted(
    MAT_ASSET_ROOT.rglob("*.mat")
)

schema_records = []
valid_mat_files = []

for mat_file in mat_files:

    print("\nInspecting:")
    print(mat_file)

    try:
        variable_info = scipy.io.whosmat(
            str(mat_file)
        )

        variable_names = {
            variable_name
            for (
                variable_name,
                shape,
                dtype,
            ) in variable_info
        }

        missing_variables = sorted(
            required_variables
            - variable_names
        )

        schema_valid = (
            len(missing_variables) == 0
        )

        if schema_valid:
            valid_mat_files.append(
                mat_file
            )

        for (
            variable_name,
            shape,
            dtype,
        ) in variable_info:

            schema_records.append({
                "mat_file": str(mat_file),
                "variable_name": variable_name,
                "shape": str(shape),
                "dtype": str(dtype),
                "required_variable": (
                    variable_name
                    in required_variables
                ),
                "file_schema_valid": schema_valid,
                "missing_required_variables": (
                    "|".join(
                        missing_variables
                    )
                ),
            })

        print("Variables:")
        for (
            variable_name,
            shape,
            dtype,
        ) in variable_info:
            print(
                "-",
                variable_name,
                shape,
                dtype,
            )

        print(
            "Required schema valid:",
            schema_valid,
        )

        if missing_variables:
            print(
                "Missing:",
                missing_variables,
            )

    except Exception as error:

        schema_records.append({
            "mat_file": str(mat_file),
            "variable_name": None,
            "shape": None,
            "dtype": None,
            "required_variable": False,
            "file_schema_valid": False,
            "missing_required_variables": (
                "INSPECTION_ERROR"
            ),
            "error": repr(error),
        })

        print(
            "Inspection failed:",
            repr(error),
        )

schema_df = pd.DataFrame(
    schema_records
)

schema_csv = (
    AUDIT_ROOT
    / "released_mat_schema.csv"
)

schema_df.to_csv(
    schema_csv,
    index=False,
)

if valid_mat_files:

    final_status = (
        "PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION"
    )

    status_explanation = (
        "At least one released .mat file contains "
        "the feature and label variables required by "
        "Tools/evaluate_norm.py."
    )

else:

    final_status = (
        "MAT_ASSET_NOT_VERIFIED"
    )

    status_explanation = (
        "No downloaded .mat file was verified with "
        "the complete evaluator schema."
    )

final_decision = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "repository": "SkyEyeLoc/MobileGeo",

    "mat_files_found": len(mat_files),

    "valid_mat_files": [
        str(path)
        for path in valid_mat_files
    ],

    "valid_mat_file_count": len(
        valid_mat_files
    ),

    "raw_image_inference_reproducible": False,

    "official_checkpoint_reproduction_supported": False,

    "final_supported_provenance": final_status,

    "explanation": status_explanation,
}

final_decision_json = (
    AUDIT_ROOT
    / "official_release_provenance.json"
)

final_decision_json.write_text(
    json.dumps(
        final_decision,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 65)
print("MOBILEGEO RELEASE PROVENANCE")
print("=" * 65)

print("\nMAT files found:")
print(len(mat_files))

print("\nValid evaluator-compatible MAT files:")
print(len(valid_mat_files))

print("\nSupported provenance:")
print(final_status)

print("\nExplanation:")
print(status_explanation)

print("\nSchema CSV:")
print(schema_csv)

print("\nFinal decision JSON:")
print(final_decision_json)


Inspecting:
/content/drive/MyDrive/mobilegeo_project/data/official_mobilegeo_precomputed_features/second_D2S.mat
Variables:
- name (1,) char
- query_name (1,) char
- gallery_name (1,) char
- gallery_f (951, 768) single
- gallery_label (1, 951) int64
- gallery_path (951,) char
- query_f (37854, 768) single
- query_label (1, 37854) int64
- query_path (37854,) char
Required schema valid: True

Inspecting:
/content/drive/MyDrive/mobilegeo_project/data/official_mobilegeo_precomputed_features/second_S2D.mat
Variables:
- name (1,) char
- query_name (1,) char
- gallery_name (1,) char
- gallery_f (51354, 768) single
- gallery_label (1, 51354) int64
- gallery_path (51354,) char
- query_f (701, 768) single
- query_label (1, 701) int64
- query_path (701,) char
Required schema valid: True

MOBILEGEO RELEASE PROVENANCE

MAT files found:
2

Valid evaluator-compatible MAT files:
2

Supported provenance:
PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION

Explanation:
At least one released .mat file contains th

In [12]:
# ============================================================
# AUDIT CELL 11
# Reproduce official precomputed-feature retrieval evaluation
#
# Evaluates:
# - second_D2S.mat
# - second_S2D.mat
#
# Saves:
# - official_precomputed_feature_metrics.csv
# - official_precomputed_feature_metrics.json
# - Top-10 rankings for both directions
#
# Important:
# This is feature-search evaluation only.
# It is NOT raw-image MobileGeo inference.
# ============================================================

from pathlib import Path
from time import perf_counter
import hashlib
import json
import csv

import numpy as np
import pandas as pd
import scipy.io
import torch

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mobilegeo_audit"
)

MAT_ASSET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "official_mobilegeo_precomputed_features"
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

mat_files = sorted(
    MAT_ASSET_ROOT.rglob("*.mat")
)

if not mat_files:
    raise FileNotFoundError(
        f"No .mat files found in:\n{MAT_ASSET_ROOT}"
    )

print("MAT files to evaluate:")

for path in mat_files:
    print("-", path.name)

# ------------------------------------------------------------
# 2. Runtime device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nEvaluation device:")
print(DEVICE)

# ------------------------------------------------------------
# 3. Helpers
# ------------------------------------------------------------

def sha256_file(
    file_path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def ensure_feature_orientation(
    features,
    labels,
    feature_name,
):
    """
    Ensure features have shape:
    number_of_samples × descriptor_dimension
    """

    features = np.asarray(
        features,
        dtype=np.float32,
    )

    labels = np.asarray(
        labels
    ).reshape(-1)

    if features.ndim != 2:
        raise ValueError(
            f"{feature_name} must be 2D, "
            f"but received {features.shape}"
        )

    if features.shape[0] == len(labels):
        return features, labels

    if features.shape[1] == len(labels):
        return features.T, labels

    raise ValueError(
        f"{feature_name} sample count does not match "
        f"its labels: features={features.shape}, "
        f"labels={len(labels)}"
    )


def safe_percentile(
    values,
    percentile,
):
    if not values:
        return None

    return float(
        np.percentile(
            np.asarray(values),
            percentile,
        )
    )

# ------------------------------------------------------------
# 4. Evaluate one MAT asset
# ------------------------------------------------------------

def evaluate_precomputed_mat(
    mat_path,
    top_k_to_save=10,
):
    mat_path = Path(mat_path)

    print("\n" + "=" * 72)
    print("Evaluating:", mat_path.name)
    print("=" * 72)

    # Load only evaluator-required variables.
    result = scipy.io.loadmat(
        str(mat_path),
        variable_names=[
            "query_f",
            "query_label",
            "gallery_f",
            "gallery_label",
        ],
    )

    required_variables = {
        "query_f",
        "query_label",
        "gallery_f",
        "gallery_label",
    }

    missing_variables = (
        required_variables
        - set(result.keys())
    )

    if missing_variables:
        raise KeyError(
            f"{mat_path.name} is missing: "
            f"{sorted(missing_variables)}"
        )

    query_features, query_labels = (
        ensure_feature_orientation(
            result["query_f"],
            result["query_label"],
            "query_f",
        )
    )

    gallery_features, gallery_labels = (
        ensure_feature_orientation(
            result["gallery_f"],
            result["gallery_label"],
            "gallery_f",
        )
    )

    if (
        query_features.shape[1]
        != gallery_features.shape[1]
    ):
        raise ValueError(
            "Query and gallery descriptor dimensions "
            "do not match."
        )

    query_count = len(query_labels)
    gallery_count = len(gallery_labels)
    descriptor_dimension = (
        query_features.shape[1]
    )

    print("Queries:", query_count)
    print("Gallery:", gallery_count)
    print(
        "Descriptor dimension:",
        descriptor_dimension,
    )

    # The released evaluator uses direct dot-product search.
    # Do not silently add normalization.
    query_norms = np.linalg.norm(
        query_features,
        axis=1,
    )

    gallery_norms = np.linalg.norm(
        gallery_features,
        axis=1,
    )

    print(
        "Query norm mean:",
        f"{query_norms.mean():.6f}",
    )

    print(
        "Gallery norm mean:",
        f"{gallery_norms.mean():.6f}",
    )

    gallery_tensor = torch.from_numpy(
        gallery_features
    ).to(DEVICE)

    # Smaller chunks for very large galleries.
    chunk_size = (
        16
        if gallery_count > 20000
        else 512
    )

    ranking_csv = (
        AUDIT_ROOT
        / (
            f"{mat_path.stem}_"
            f"top{top_k_to_save}_rankings.csv"
        )
    )

    ranking_columns = [
        "provenance_label",
        "evaluation_type",
        "asset_name",
        "query_index",
        "query_label",
        "candidate_rank",
        "gallery_index",
        "gallery_label",
        "score",
        "is_positive",
    ]

    recall_1_count = 0
    recall_5_count = 0
    recall_10_count = 0

    ap_values = []
    first_positive_ranks = []

    valid_query_count = 0
    query_without_positive_count = 0

    evaluation_start = perf_counter()

    with open(
        ranking_csv,
        "w",
        newline="",
        encoding="utf-8",
    ) as ranking_file:

        ranking_writer = csv.DictWriter(
            ranking_file,
            fieldnames=ranking_columns,
        )

        ranking_writer.writeheader()

        with torch.no_grad():

            for start_index in range(
                0,
                query_count,
                chunk_size,
            ):
                end_index = min(
                    start_index + chunk_size,
                    query_count,
                )

                query_chunk = torch.from_numpy(
                    query_features[
                        start_index:end_index
                    ]
                ).to(DEVICE)

                score_chunk = (
                    query_chunk
                    @ gallery_tensor.T
                )

                sorted_scores, sorted_indices = (
                    torch.sort(
                        score_chunk,
                        dim=1,
                        descending=True,
                    )
                )

                sorted_scores = (
                    sorted_scores
                    .detach()
                    .cpu()
                    .numpy()
                )

                sorted_indices = (
                    sorted_indices
                    .detach()
                    .cpu()
                    .numpy()
                )

                for local_index in range(
                    end_index - start_index
                ):
                    query_index = (
                        start_index
                        + local_index
                    )

                    query_label = (
                        query_labels[
                            query_index
                        ]
                    )

                    ranked_gallery_indices = (
                        sorted_indices[
                            local_index
                        ]
                    )

                    ranked_scores = (
                        sorted_scores[
                            local_index
                        ]
                    )

                    ranked_gallery_labels = (
                        gallery_labels[
                            ranked_gallery_indices
                        ]
                    )

                    relevance = (
                        ranked_gallery_labels
                        == query_label
                    )

                    positive_count = int(
                        relevance.sum()
                    )

                    if positive_count == 0:
                        query_without_positive_count += 1
                        continue

                    valid_query_count += 1

                    positive_positions = (
                        np.flatnonzero(
                            relevance
                        )
                    )

                    first_positive_rank = int(
                        positive_positions[0] + 1
                    )

                    first_positive_ranks.append(
                        first_positive_rank
                    )

                    if first_positive_rank <= 1:
                        recall_1_count += 1

                    if first_positive_rank <= 5:
                        recall_5_count += 1

                    if first_positive_rank <= 10:
                        recall_10_count += 1

                    cumulative_positives = (
                        np.cumsum(
                            relevance.astype(
                                np.float64
                            )
                        )
                    )

                    rank_numbers = np.arange(
                        1,
                        len(relevance) + 1,
                        dtype=np.float64,
                    )

                    precision_at_ranks = (
                        cumulative_positives
                        / rank_numbers
                    )

                    average_precision = float(
                        precision_at_ranks[
                            relevance
                        ].sum()
                        / positive_count
                    )

                    ap_values.append(
                        average_precision
                    )

                    saved_k = min(
                        top_k_to_save,
                        gallery_count,
                    )

                    for candidate_offset in range(
                        saved_k
                    ):
                        gallery_index = int(
                            ranked_gallery_indices[
                                candidate_offset
                            ]
                        )

                        gallery_label = (
                            gallery_labels[
                                gallery_index
                            ]
                        )

                        ranking_writer.writerow({
                            "provenance_label": (
                                "PUBLISHED_PRECOMPUTED_"
                                "FEATURE_EVALUATION"
                            ),
                            "evaluation_type": (
                                "FEATURE_SEARCH_ONLY"
                            ),
                            "asset_name": (
                                mat_path.name
                            ),
                            "query_index": (
                                query_index
                            ),
                            "query_label": (
                                int(query_label)
                            ),
                            "candidate_rank": (
                                candidate_offset + 1
                            ),
                            "gallery_index": (
                                gallery_index
                            ),
                            "gallery_label": (
                                int(gallery_label)
                            ),
                            "score": float(
                                ranked_scores[
                                    candidate_offset
                                ]
                            ),
                            "is_positive": bool(
                                gallery_label
                                == query_label
                            ),
                        })

                print(
                    f"Processed queries: "
                    f"{end_index}/{query_count}",
                    end="\r",
                )

    evaluation_seconds = (
        perf_counter()
        - evaluation_start
    )

    if valid_query_count == 0:
        raise RuntimeError(
            f"No valid queries in {mat_path.name}"
        )

    metrics_record = {
        "asset_name": mat_path.name,
        "asset_path": str(mat_path),
        "asset_sha256": sha256_file(
            mat_path
        ),
        "provenance_label": (
            "PUBLISHED_PRECOMPUTED_"
            "FEATURE_EVALUATION"
        ),
        "evaluation_type": (
            "FEATURE_SEARCH_ONLY"
        ),
        "raw_image_inference": False,
        "official_checkpoint_loaded": False,
        "search_function": (
            "DIRECT_DOT_PRODUCT"
        ),
        "query_count": int(
            query_count
        ),
        "gallery_count": int(
            gallery_count
        ),
        "valid_query_count": int(
            valid_query_count
        ),
        "queries_without_positive": int(
            query_without_positive_count
        ),
        "descriptor_dimension": int(
            descriptor_dimension
        ),
        "query_norm_mean": float(
            query_norms.mean()
        ),
        "query_norm_std": float(
            query_norms.std()
        ),
        "gallery_norm_mean": float(
            gallery_norms.mean()
        ),
        "gallery_norm_std": float(
            gallery_norms.std()
        ),
        "recall_at_1_percent": float(
            100.0
            * recall_1_count
            / valid_query_count
        ),
        "recall_at_5_percent": float(
            100.0
            * recall_5_count
            / valid_query_count
        ),
        "recall_at_10_percent": float(
            100.0
            * recall_10_count
            / valid_query_count
        ),
        "mean_average_precision_percent": float(
            100.0
            * np.mean(ap_values)
        ),
        "median_first_positive_rank": (
            safe_percentile(
                first_positive_ranks,
                50,
            )
        ),
        "p95_first_positive_rank": (
            safe_percentile(
                first_positive_ranks,
                95,
            )
        ),
        "evaluation_seconds": float(
            evaluation_seconds
        ),
        "mean_search_ms_per_query": float(
            evaluation_seconds
            * 1000.0
            / query_count
        ),
        "timing_scope": (
            "FEATURE_LOADING_EXCLUDED;"
            "DOT_PRODUCT_SORT_AND_METRICS_INCLUDED"
        ),
        "top_k_ranking_csv": str(
            ranking_csv
        ),
    }

    print("\n\nResults:")
    print(
        "Recall@1:",
        f"{metrics_record['recall_at_1_percent']:.2f}%",
    )

    print(
        "Recall@5:",
        f"{metrics_record['recall_at_5_percent']:.2f}%",
    )

    print(
        "Recall@10:",
        f"{metrics_record['recall_at_10_percent']:.2f}%",
    )

    print(
        "mAP:",
        (
            f"{metrics_record['mean_average_precision_percent']:.2f}%"
        ),
    )

    print(
        "Evaluation seconds:",
        f"{evaluation_seconds:.3f}",
    )

    print("Top-K rankings:")
    print(ranking_csv)

    return metrics_record

# ------------------------------------------------------------
# 5. Evaluate all released MAT files
# ------------------------------------------------------------

all_metrics = []

for mat_path in mat_files:
    all_metrics.append(
        evaluate_precomputed_mat(
            mat_path,
            top_k_to_save=10,
        )
    )

# ------------------------------------------------------------
# 6. Save combined metrics
# ------------------------------------------------------------

metrics_df = pd.DataFrame(
    all_metrics
)

metrics_csv = (
    AUDIT_ROOT
    / "official_precomputed_feature_metrics.csv"
)

metrics_json = (
    AUDIT_ROOT
    / "official_precomputed_feature_metrics.json"
)

metrics_df.to_csv(
    metrics_csv,
    index=False,
)

metrics_json.write_text(
    json.dumps(
        all_metrics,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n" + "=" * 72)
print("OFFICIAL PRECOMPUTED FEATURE EVALUATION COMPLETE")
print("=" * 72)

print(
    "\n",
    metrics_df[
        [
            "asset_name",
            "query_count",
            "gallery_count",
            "descriptor_dimension",
            "recall_at_1_percent",
            "recall_at_5_percent",
            "recall_at_10_percent",
            "mean_average_precision_percent",
        ]
    ].to_string(
        index=False
    ),
)

print("\nMetrics CSV:")
print(metrics_csv)

print("\nMetrics JSON:")
print(metrics_json)

MAT files to evaluate:
- second_D2S.mat
- second_S2D.mat

Evaluation device:
cuda

Evaluating: second_D2S.mat
Queries: 37854
Gallery: 951
Descriptor dimension: 768
Query norm mean: 1.000000
Gallery norm mean: 1.000000


Results:
Recall@1: 93.95%
Recall@5: 98.02%
Recall@10: 98.57%
mAP: 95.83%
Evaluation seconds: 5.193
Top-K rankings:
/content/drive/MyDrive/mobilegeo_project/results/mobilegeo_audit/second_D2S_top10_rankings.csv

Evaluating: second_S2D.mat
Queries: 701
Gallery: 51354
Descriptor dimension: 768
Query norm mean: 1.000000
Gallery norm mean: 1.000000


Results:
Recall@1: 95.72%
Recall@5: 97.15%
Recall@10: 97.43%
mAP: 92.63%
Evaluation seconds: 0.772
Top-K rankings:
/content/drive/MyDrive/mobilegeo_project/results/mobilegeo_audit/second_S2D_top10_rankings.csv

OFFICIAL PRECOMPUTED FEATURE EVALUATION COMPLETE

     asset_name  query_count  gallery_count  descriptor_dimension  recall_at_1_percent  recall_at_5_percent  recall_at_10_percent  mean_average_precision_percent
second_D2

In [13]:
# ============================================================
# AUDIT CELL 12
# Finalize Phase 1 release audit and package evidence
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import zipfile
import hashlib
import pandas as pd

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/mobilegeo_project"
)

AUDIT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "mobilegeo_audit"
)

METRICS_JSON = (
    AUDIT_ROOT
    / "official_precomputed_feature_metrics.json"
)

PROVENANCE_JSON = (
    AUDIT_ROOT
    / "official_release_provenance.json"
)

FINAL_REPORT = (
    AUDIT_ROOT
    / "phase1_official_release_audit_report.md"
)

FINAL_SUMMARY_JSON = (
    AUDIT_ROOT
    / "phase1_official_release_audit_summary.json"
)

AUDIT_ZIP = (
    PROJECT_ROOT
    / "reports"
    / "mobilegeo_phase1_official_release_audit.zip"
)

if not METRICS_JSON.is_file():
    raise FileNotFoundError(
        "Run AUDIT CELL 11 first."
    )

if not PROVENANCE_JSON.is_file():
    raise FileNotFoundError(
        "AUDIT CELL 10 output is missing."
    )

# ------------------------------------------------------------
# 2. Load verified evidence
# ------------------------------------------------------------

metrics = json.loads(
    METRICS_JSON.read_text(
        encoding="utf-8"
    )
)

provenance = json.loads(
    PROVENANCE_JSON.read_text(
        encoding="utf-8"
    )
)

final_status = (
    "PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION"
)

summary = {
    "completed_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "phase": (
        "PHASE_1_OFFICIAL_MOBILEGEO_RELEASE_AUDIT"
    ),

    "phase_complete": True,

    "repository": "SkyEyeLoc/MobileGeo",

    "supported_provenance": final_status,

    "raw_image_inference_reproduced": False,

    "official_mobilegeo_checkpoint_found": False,

    "precomputed_feature_evaluation_completed": True,

    "valid_mat_asset_count": int(
        provenance.get(
            "valid_mat_file_count",
            0,
        )
    ),

    "evaluated_assets": [
        {
            "asset_name": row["asset_name"],
            "asset_sha256": row[
                "asset_sha256"
            ],
            "query_count": row[
                "query_count"
            ],
            "gallery_count": row[
                "gallery_count"
            ],
            "descriptor_dimension": row[
                "descriptor_dimension"
            ],
            "recall_at_1_percent": row[
                "recall_at_1_percent"
            ],
            "recall_at_5_percent": row[
                "recall_at_5_percent"
            ],
            "recall_at_10_percent": row[
                "recall_at_10_percent"
            ],
            "mean_average_precision_percent": row[
                "mean_average_precision_percent"
            ],
        }
        for row in metrics
    ],

    "limitations": [
        (
            "No trained MobileGeo checkpoint was "
            "verified in the cloned repository."
        ),
        (
            "No complete raw-image preprocessing and "
            "query/gallery inference pipeline was reproduced."
        ),
        (
            "The released evaluation operates on "
            "precomputed descriptors."
        ),
        (
            "The repository MobileGeo weight link "
            "was empty during the audit."
        ),
        (
            "No repository license file was verified."
        ),
        (
            "AGX Orin performance claims were not "
            "independently reproduced."
        ),
    ],

    "next_phase": (
        "PHASE_2_CORRECTED_SUES200_"
        "LOCATION_DISJOINT_BENCHMARK"
    ),
}

FINAL_SUMMARY_JSON.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

# ------------------------------------------------------------
# 3. Build readable result table
# ------------------------------------------------------------

result_lines = []

for row in metrics:
    result_lines.append(
        "| "
        + row["asset_name"]
        + " | "
        + str(row["query_count"])
        + " | "
        + str(row["gallery_count"])
        + " | "
        + str(row["descriptor_dimension"])
        + " | "
        + f"{row['recall_at_1_percent']:.2f}%"
        + " | "
        + f"{row['recall_at_5_percent']:.2f}%"
        + " | "
        + f"{row['recall_at_10_percent']:.2f}%"
        + " | "
        + (
            f"{row['mean_average_precision_percent']:.2f}%"
        )
        + " |"
    )

result_table = "\n".join(
    result_lines
)

# ------------------------------------------------------------
# 4. Write final Phase 1 report
# ------------------------------------------------------------

report_text = f"""# MobileGeo Phase 1 Official Release Audit

## Final supported provenance

`{final_status}`

## What was reproduced

The released MobileGeo `.mat` descriptor files were validated
and evaluated using direct dot-product retrieval consistent with
the released `Tools/evaluate_norm.py` evaluator.

## Evaluated assets

| Asset | Queries | Gallery | Descriptor dimension | R@1 | R@5 | R@10 | mAP |
|---|---:|---:|---:|---:|---:|---:|---:|
{result_table}

## What was not reproduced

- Raw-image MobileGeo inference
- An official trained MobileGeo checkpoint
- A complete query/gallery image-preprocessing pipeline
- MobileGeo training
- AGX Orin runtime results
- Jetson Orin Nano runtime results

## Repository findings

- Two valid evaluator-compatible `.mat` files were released.
- The evaluator requires `query_f`, `query_label`,
  `gallery_f`, and `gallery_label`.
- The released descriptors have dimension 768.
- The MobileGeo weights link in the current README is empty.
- Generic ConvNeXt pretrained backbone URLs are not
  MobileGeo model checkpoints.
- No repository licence file was verified.
- `README.md~` contains older PFED documentation and was
  treated only as repository-history evidence.

## Reporting restriction

These metrics must be labelled:

`PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION`

They must not be labelled:

- `OFFICIAL_CHECKPOINT_REPRODUCTION`
- raw-image inference
- verified GPS-denied positioning
- verified global UAV pose

## Phase 1 decision

Phase 1 is complete at the strongest provenance supported by
the released evidence.

## Next phase

Create and validate the corrected SUES-200 location-disjoint
benchmark in:

`02_sues200_location_disjoint_benchmark.ipynb`
"""

FINAL_REPORT.write_text(
    report_text,
    encoding="utf-8",
)

# ------------------------------------------------------------
# 5. Create evidence ZIP
# ------------------------------------------------------------

AUDIT_ZIP.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if AUDIT_ZIP.is_file():
    AUDIT_ZIP.unlink()

with zipfile.ZipFile(
    AUDIT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for file_path in sorted(
        AUDIT_ROOT.rglob("*")
    ):
        if not file_path.is_file():
            continue

        archive.write(
            file_path,
            arcname=str(
                Path("mobilegeo_phase1_audit")
                / file_path.relative_to(
                    AUDIT_ROOT
                )
            ),
        )

# ------------------------------------------------------------
# 6. ZIP checksum
# ------------------------------------------------------------

def sha256_file(
    file_path,
):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


zip_sha256 = sha256_file(
    AUDIT_ZIP
)

checksum_path = (
    AUDIT_ZIP.with_suffix(
        ".zip.sha256"
    )
)

checksum_path.write_text(
    f"{zip_sha256}  {AUDIT_ZIP.name}\n",
    encoding="utf-8",
)

# ------------------------------------------------------------
# 7. Validate ZIP
# ------------------------------------------------------------

with zipfile.ZipFile(
    AUDIT_ZIP,
    "r",
) as archive:

    corrupt_file = archive.testzip()
    archived_files = archive.namelist()

if corrupt_file is not None:
    raise RuntimeError(
        f"Corrupt ZIP member: {corrupt_file}"
    )

# ------------------------------------------------------------
# 8. Final output
# ------------------------------------------------------------

print("=" * 70)
print("✅ PHASE 1 MOBILEGEO RELEASE AUDIT COMPLETE")
print("=" * 70)

print("\nFinal provenance:")
print(final_status)

print("\nRaw-image inference reproduced:")
print(False)

print("\nOfficial checkpoint reproduced:")
print(False)

print("\nPrecomputed-feature evaluation:")
print(True)

print("\nFinal report:")
print(FINAL_REPORT)

print("\nSummary JSON:")
print(FINAL_SUMMARY_JSON)

print("\nAudit evidence ZIP:")
print(AUDIT_ZIP)

print("\nFiles inside ZIP:")
print(len(archived_files))

print("\nZIP SHA-256:")
print(zip_sha256)

print("\nNext notebook:")
print(
    PROJECT_ROOT
    / "notebooks"
    / "02_sues200_location_disjoint_benchmark.ipynb"
)

✅ PHASE 1 MOBILEGEO RELEASE AUDIT COMPLETE

Final provenance:
PUBLISHED_PRECOMPUTED_FEATURE_EVALUATION

Raw-image inference reproduced:
False

Official checkpoint reproduced:
False

Precomputed-feature evaluation:
True

Final report:
/content/drive/MyDrive/mobilegeo_project/results/mobilegeo_audit/phase1_official_release_audit_report.md

Summary JSON:
/content/drive/MyDrive/mobilegeo_project/results/mobilegeo_audit/phase1_official_release_audit_summary.json

Audit evidence ZIP:
/content/drive/MyDrive/mobilegeo_project/reports/mobilegeo_phase1_official_release_audit.zip

Files inside ZIP:
19

ZIP SHA-256:
acd77eec6d60e967ab86179a5b308f82fcf67d0393e975fbc6452af4b56fbe54

Next notebook:
/content/drive/MyDrive/mobilegeo_project/notebooks/02_sues200_location_disjoint_benchmark.ipynb


## Work cells

Add the phase-specific implementation below this cell.
